In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

train_path = os.path.join(path,"PlantVillage","train")
test_path = os.path.join(path,"PlantVillage","test")

transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])


train_dataset = ImageFolder(root=train_path, transform=transform)
test_dataset  = ImageFolder(root=test_path,  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")


In [ ]:
# Write your code here
import torch
import torch.nn as nn

class PotatoModel(nn.Module):
    def __init__(self):
        """
        1️⃣ Define all layers in the model.
        """
        super(PotatoModel, self).__init__()

        self.features = nn.Sequential( #1
           nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
           nn.ReLU(),
           nn.BatchNorm2d(16),
            #2
           nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
           nn.ReLU(),
           nn.BatchNorm2d(32),
            #3
           nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
           nn.ReLU(),
           nn.BatchNorm2d(64),

            #4
           nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
           nn.ReLU(),
           nn.BatchNorm2d(128),

           #5
           nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1),
           nn.ReLU(),
           nn.BatchNorm2d(256),

        )



        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear( 32 *32 * 256 ,128),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Linear(128, 26),
            )
    def forward(self, x):
        """
        2️⃣ Define the forward pass (how data flows through the model).
        """
        x = self.features(x)
        print(x.shape)


        x = self.classifier(x)



        return x


In [ ]:
# Write your code here

from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PotatoModel().to(device)

def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=1)
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        # TO-DO: Zero the gradients
        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        # TO-DO: Backward pass
        loss.backward()

        # TO-DO: Update parameters
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), labels)

    return total_loss / len(loader), total_acc / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images, labels = images.to(device), labels.to(device)

            logits = model(images)

            loss = criterion(logits, labels)

            total_loss += loss.item()
            total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)



In [ ]:
# Write your code here
import torch.optim as optim


criterion = nn.CrossEntropyLoss()

learning_rate = 0.0001

optimizer = optim.AdamW(model.parameters(),lr = learning_rate)

num_epochs = 10  # How many times to iterate through the dataset?

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    # Store history
    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


In [ ]:
# Write your code here
